In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import torch 
import time
import datetime

In [47]:
surcharge_amount = 0.5
# Load a CSV file named 'data.csv' into a DataFrame
items = pd.read_csv('items.csv')
sales = pd.read_csv('sales.csv')

items_list = list(items["item_name"])
items_list

['Espresso',
 'Americano',
 'Latte',
 'Cappuccino',
 'Flat White',
 'Mocha',
 'Iced Coffee',
 'Cold Brew',
 'Iced Latte',
 'Iced Matcha Latte',
 'Matcha Latte',
 'Chai Latte',
 'Green Tea',
 'Hot Chocolate',
 'Reusable Coffee Cup',
 'Tote Bag',
 'Coffee Beans (1 lb)']

In [48]:
with_prices = pd.merge(sales, items, on='item_name', how='inner')
with_prices.iloc[0]

date                 2022-01-01
time                   07:03:30
item_name             Cold Brew
transaction_type        Takeout
own_cup                    True
surcharge                 False
customer_id               26946
price                      4.75
production_cost            1.35
item_type                 Drink
drink_temperature          Cold
drink_type               Coffee
Name: 0, dtype: object

In [50]:
def normalize(row): 

    assert row["item_type"] == "Drink"

    x = time.strptime(row["time"],'%H:%M:%S')
    normalized_time = datetime.timedelta(hours=x.tm_hour,minutes=x.tm_min,seconds=x.tm_sec).total_seconds() / (24 * 60 * 60)

    normalized_date = time.strptime(row["date"],'%Y-%M-%d').tm_yday / 366

    normalized_item = items_list.index(row["item_name"]) / (len(items) - 1)

    return (normalized_time, normalized_item, normalized_date)
    

normalize(with_prices.iloc[0])

(0.29409722222222223, 0.4375, 0.00273224043715847)

In [53]:


class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data[data["item_type"] == "Drink"]

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        return normalize(self.data.iloc[idx]), float(bool(self.data.iloc[idx]["own_cup"]))

0.0

Ran out of time so I chatGPTed it, I could do it myself, but i gotta run and I thought I would leave you with something. 

In [ ]:
import torch
from torch.utils.data import Dataset
import time, datetime

def normalize(row):
    assert row["item_type"] == "Drink"

    # ----- Normalize time (HH:MM:SS) -----
    t = time.strptime(row["time"], '%H:%M:%S')
    seconds = datetime.timedelta(
        hours=t.tm_hour, minutes=t.tm_min, seconds=t.tm_sec
    ).total_seconds()
    normalized_time = seconds / (24 * 60 * 60)

    # ----- Normalize date (YYYY-MM-DD) -----
    d = time.strptime(row["date"], '%Y-%m-%d')   # FIXED: %m not %M
    normalized_date = d.tm_yday / 366

    # ----- Normalize item type (index / max index) -----
    normalized_item = items_list.index(row["item_name"]) / (len(items_list) - 1)

    return (
        normalized_time,
        normalized_item,
        normalized_date
    )

class CustomDataset(Dataset):
    def __init__(self, data):
        # Keep only drinks once here
        self.data = data[data["item_type"] == "Drink"].reset_index(drop=True)

    def __len__(self):
        return len(self.data)      # FIXED

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        x = normalize(row)
        x = torch.tensor(x, dtype=torch.float32)     # convert to tensor

        y = float(bool(row["own_cup"]))
        y = torch.tensor(y, dtype=torch.float32)     # tensor target

        return x, y


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam

# -------------------------------------------------------------
# 1) Build the full dataset from your single DataFrame
# -------------------------------------------------------------
full_dataset = CustomDataset(with_prices)   # your single pandas dataframe

# built-in PyTorch random split
train_size = int(0.8 * len(full_dataset))
test_size  = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# -------------------------------------------------------------
# 2) Model (3 input features from normalize())
# -------------------------------------------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = Net()

# -------------------------------------------------------------
# 3) Loss, optimizer, device
# -------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr=1e-3)

# -------------------------------------------------------------
# 4) Training loop
# -------------------------------------------------------------
for epoch in range(10):

    # ---- TRAIN ----
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ---- TEST ----
    model.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device).unsqueeze(1)

            preds = model(x)
            test_loss += criterion(preds, y).item()

            predicted = (preds > 0.5).float()
            correct += (predicted == y).sum().item()
            total += y.size(0)

    print(
        f"Epoch {epoch+1} | "
        f"train_loss={total_loss/len(train_loader):.4f} | "
        f"test_loss={test_loss/len(test_loader):.4f} | "
        f"test_acc={correct/total:.4f}"
    )


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'CustomDataset' on <module '__main__' (built-in)>
